In [1]:
import os
REPO_NAME = "vision_blindspots"
REPO_URL = "https://github.com/aprzezdziecka/vision_blindspots.git"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

/content/vision_blindspots


In [2]:
from transformers import CLIPProcessor, CLIPModel, AutoProcessor, AutoModel
from PIL import Image
import requests
from datasets import load_dataset
from huggingface_hub import notebook_login
import torch
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm
import pandas as pd

In [6]:

# funkcja pomocnicza która przyjmuje csv i liczbę ile ma zwrócić zdjęć
# ma zwrocić tą liczbę par ile chcemy

# mamy tak 50 zdjęć, piszemy do blipa żeby definiował kategorie w jakich się różni
# 5-7 powtarzających się kategorii w jakich różnią się zdjęcia
# zastanowić się czy dajemy sugestie tych kategorii

# ręczne dopisanie kategorii które według nas brakuje

# na pełnej bazie teraz inny prompt i pytamy czy dana para się różni w danych kategoriach + opis różnic



In [3]:
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
# pliki = [
#     "/content/vision_blindspots/clip_caltech_blind_spots.csv",
#     "/content/vision_blindspots/dino_caltech_blind_spots.csv",
#     "/content/vision_blindspots/siglip2_caltech_blind_spots.csv",
#     "/content/vision_blindspots/siglip2_imagenet_test_blind_spots.csv",
#     "/content/vision_blindspots/dino_imagenet_test_blind_spots.csv",
#     "/content/vision_blindspots/clip_imagenet_test_blind_spots.csv",
#     "/content/vision_blindspots/siglip_caltech_blind_spots.csv",
#     "/content/vision_blindspots/siglip_imagenet_test_blind_spots.csv"
# ]

In [ ]:
import pandas as pd
import torch
import os
from tqdm.auto import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from datasets import load_dataset
from PIL import Image # DODANE: do manipulacji i sklejania obrazów

# --- 1. Inicjalizacja modelu ---
print("Ładowanie modelu BLIP-2 do VRAM...")
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", device_map="auto")

# --- 2. Ładowanie zbiorów ---
print("Pobieranie i ładowanie zbiorów do pamięci podręcznej...")
dataset_caltech = load_dataset("flwrlabs/caltech101", split="train", trust_remote_code=True)

def wybierz_dataset_na_podstawie_nazwy(sciezka_pliku):
    nazwa_pliku = os.path.basename(sciezka_pliku).lower()
    if "caltech" in nazwa_pliku:
        return dataset_caltech, "Caltech101"
    else:
        raise ValueError(f"Nie rozpoznano zbioru: {nazwa_pliku}")

# --- DODANE: Funkcja do sklejania dwóch zdjęć w poziomie ---
def sklej_zdjecia_w_poziomie(img1, img2):
    """
    Skleja dwa zdjęcia (PIL.Image) obok siebie.
    Dopasowuje wysokość drugiego zdjęcia do pierwszego, aby nie było pustych przestrzeni.
    """
    w1, h1 = img1.size
    w2, h2 = img2.size

    # Skalujemy drugie zdjęcie, aby miało taką samą wysokość jak pierwsze
    nowa_szerokosc_img2 = int((h1 / h2) * w2)
    img2_przeskalowane = img2.resize((nowa_szerokosc_img2, h1))

    # Tworzymy nowe, puste "płótno" o łącznej szerokości obu zdjęć
    calkowita_szerokosc = w1 + nowa_szerokosc_img2
    sklejony_obraz = Image.new('RGB', (calkowita_szerokosc, h1))

    # Wklejamy oba zdjęcia na płótno
    sklejony_obraz.paste(img1, (0, 0)) # Pierwsze od lewej (0,0)
    sklejony_obraz.paste(img2_przeskalowane, (w1, 0)) # Drugie wklejamy za pierwszym

    return sklejony_obraz


# --- 3. Analiza Wsadowa (Batch Processing) ---
def przeanalizuj_csv_w_paczkach(sciezka_csv, pytanie, batch_size=8):
    print(f"\n--- Szybka analiza dla pliku: {os.path.basename(sciezka_csv)} ---")
    df = pd.read_csv(sciezka_csv, sep=';', index_col=0)

    odpowiedni_dataset, nazwa_zbioru = wybierz_dataset_na_podstawie_nazwy(sciezka_csv)
    print(f"Zbiór: {nazwa_zbioru}. Rozpoczynam inferencję na GPU...")

    wyniki = []

    for i in tqdm(range(0, len(df), batch_size), desc="Przetwarzanie paczek (Batches)"):
        paczka_df = df.iloc[i : i + batch_size]

        obrazki_w_paczce = []
        pary_id = [] # Zmienione: przechowujemy pary jako krotki (tuple)

        # 1. Zbieranie i sklejanie zdjęć z paczki
        for _, row in paczka_df.iterrows():
            id_1 = int(row['id_1'])
            id_2 = int(row['id_2'])

            img_1 = odpowiedni_dataset[id_1]['image'].convert('RGB')
            img_2 = odpowiedni_dataset[id_2]['image'].convert('RGB')

            # SKLEJANIE ZDJĘĆ
            sklejone_zdjecie = sklej_zdjecia_w_poziomie(img_1, img_2)

            obrazki_w_paczce.append(sklejone_zdjecie) # Dodajemy JEDNO sklejone zdjęcie
            pary_id.append((id_1, id_2))

        # 2. Wysyłanie paczki sklejonych zdjęć
        pytania_w_paczce = [pytanie] * len(obrazki_w_paczce)

        inputs = processor(images=obrazki_w_paczce, text=pytania_w_paczce, padding=True, return_tensors="pt").to("cuda")

        # 3. Inferencja
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=100)

        # 4. Dekodowanie odpowiedzi
        odpowiedzi = processor.batch_decode(out, skip_special_tokens=True)

        # 5. Zapis wyników - teraz każda odpowiedź odpowiada jednej parze
        for j, odp in enumerate(odpowiedzi):
            id_1, id_2 = pary_id[j]
            wyniki.append({
                'id_1': id_1,
                'id_2': id_2,
                'odpowiedz': odp.strip()
            })

    return pd.DataFrame(wyniki)

# --- 4. Uruchomienie ---
wybrany_plik = "/content/vision_blindspots/clip_caltech_blind_spots.csv"

# ZMIENIONE: Pytanie sugerujące, że na obrazku są dwa zdjęcia obok siebie
#pytanie_testowe = "Question: This image contains two pictures side by side. What is the difference between the picture on the left and the picture on the right? Answer:"
# pytanie_testowe = """Question: This image shows two photos placed side by side. Compare the left photo and the right photo across these categories:
# 1. Subject/Object: Are the main subjects the same or different?
# 2. Color: Are the colors similar or different?
# 3. Perspective/Angle: Is the camera angle or viewpoint the same or different?
# 4. Background: Is the background the same or different?
# 5. Overall: Are the two photos the SAME or DIFFERENT?
# Answer:"""
pytanie_testowe = "Question: This image shows two photos side by side. Compare them: same subject? same colors? same angle? same background? same size? Are they overall same or different? Answer:"
# Jeśli sklejasz zdjęcia, jedno wywołanie to teraz MIEJ pamięci w batchu (bo wysyłasz 8 sklejonych obrazów, a nie 16 osobnych),
# więc prawdopodobnie bez problemu możesz utrzymać batch_size=8 lub nawet go zwiększyć.
df_wynikowy = przeanalizuj_csv_w_paczkach(wybrany_plik, pytanie_testowe, batch_size=8)

print("\nGotowe! Oto pierwsze 5 wyników:")
print(df_wynikowy.head())

Ładowanie modelu BLIP-2 do VRAM...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'flwrlabs/caltech101' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'flwrlabs/caltech101' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Pobieranie i ładowanie zbiorów do pamięci podręcznej...

--- Szybka analiza dla pliku: clip_caltech_blind_spots.csv ---
Zbiór: Caltech101. Rozpoczynam inferencję na GPU...


Przetwarzanie paczek (Batches):   0%|          | 0/25 [00:00<?, ?it/s]

In [4]:
df_wynikowy

,id_1,id_2,odpowiedz
0,20,465,Question: This image shows two photos side by ...
1,55,7297,Question: This image shows two photos side by ...
2,84,8550,Question: This image shows two photos side by ...
3,84,4583,Question: This image shows two photos side by ...
4,116,5039,Question: This image shows two photos side by ...
...,...,...,...
189,3609,8550,Question: This image shows two photos side by ...
190,1732,8550,Question: This image shows two photos side by ...
191,3949,8550,Question: This image shows two photos side by ...
192,6005,8663,Question: This image shows two photos side by ...
